In [2]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy

In [2]:
import pandas as pd
import numpy as np


In [4]:

# ---------------------------------------------------------
# 1. LOAD RAW SCOPUS SOURCE TITLE LIST
# ---------------------------------------------------------

# Read the Excel file exported from Scopus
scopus_raw = pd.read_excel("scopus_list_Oct_2025.xlsx")

# Quick look at the columns (optional, helps debugging)
print("Columns in raw Scopus file:")
print(scopus_raw.columns.tolist())


# ---------------------------------------------------------
# 2. NORMALIZE COLUMN NAMES (OPTIONAL BUT HELPFUL)
# ---------------------------------------------------------

# Strip leading/trailing spaces from column names to avoid issues
scopus_raw.columns = scopus_raw.columns.str.strip()

# For convenience, define the exact column names we will use
# (they must match what you see in print(scopus_raw.columns))
COL_SOURCE_TITLE = "Source Title"
COL_ISSN         = "ISSN"
COL_EISSN        = "EISSN"
COL_ACTIVE       = "Active or Inactive"
COL_TYPE         = "Source Type"
COL_PUBLISHER    = "Publisher"
COL_ASJC         = "All Science Journal Classification Codes (ASJC)"


# ---------------------------------------------------------
# 3. FUNCTION TO NORMALIZE ISSN STRINGS
# ---------------------------------------------------------

def norm_issn(x):
    """
    Normalize ISSN:
    - if value is NaN → return NaN
    - convert to string
    - remove hyphens
    - strip surrounding spaces
    - lowercase (so 'X' becomes 'x')
    """
    if pd.isna(x):
        return np.nan
    return str(x).replace("-", "").strip().lower()


# ---------------------------------------------------------
# 4. CREATE NORMALIZED ISSN COLUMNS FOR ISSN AND EISSN
#    (ALIGNING WITH: "Create a normalized ISSN column")
# ---------------------------------------------------------

# Apply normalization to the ISSN column
scopus_raw["ISSN_norm"] = scopus_raw[COL_ISSN].apply(norm_issn)

# Apply normalization to the EISSN column
scopus_raw["EISSN_norm"] = scopus_raw[COL_EISSN].apply(norm_issn)


# ---------------------------------------------------------
# 5. SELECT CORE COLUMNS WE CARE ABOUT
#    (ALIGNING WITH: "Keep columns: ISSN, EISSN, Source Title, Type, etc.")
# ---------------------------------------------------------

# These columns describe the journal/venue
id_cols = [
    COL_SOURCE_TITLE,
    COL_TYPE,
    COL_ACTIVE,
    COL_PUBLISHER,
    COL_ASJC,
]

# These are the normalized ISSN variants we will melt to long format
issn_norm_cols = ["ISSN_norm", "EISSN_norm"]


# ---------------------------------------------------------
# 6. RESHAPE TO "ONE ISSN PER ROW" (LONG FORMAT)
#    (ALIGNING WITH STEP 3 OF MERGE WORKFLOW: "Explode to one ISSN per row")
# ---------------------------------------------------------

# Melt the table so that each row has exactly one normalized ISSN
scopus_long = scopus_raw.melt(
    id_vars=id_cols,            # columns we keep as identifiers
    value_vars=issn_norm_cols,  # columns we melt (ISSN_norm + EISSN_norm)
    var_name="issn_variant",    # name of the column indicating ISSN vs EISSN
    value_name="issn_norm"      # name of the column storing the normalized ISSN
)

# Drop rows where we still don't have an ISSN (i.e., ISSN and EISSN were both missing)
scopus_long = scopus_long.dropna(subset=["issn_norm"])


# ---------------------------------------------------------
# 7. CLEAN "ACTIVE OR INACTIVE" FIELD INTO BOOLEAN
# ---------------------------------------------------------

# Convert "Active or Inactive" to a simple boolean:

scopus_long["active"] = (
    scopus_long[COL_ACTIVE]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("active")   # True only if the value is exactly "active"
)

# ---------------------------------------------------------
# 8. HANDLE MISSING PUBLISHER / ASJC VALUES
# ---------------------------------------------------------

# Fill missing publishers with a placeholder (instead of NaN)
scopus_long[COL_PUBLISHER] = scopus_long[COL_PUBLISHER].fillna("unknown")

# Keep ASJC raw as a string; if NaN, fill with empty string
scopus_long[COL_ASJC] = scopus_long[COL_ASJC].fillna("")


# ---------------------------------------------------------
# 9. BUILD FINAL SCOPUS LOOKUP TABLE
#    (ONE ROW PER ISSN + VENUE METADATA)
# ---------------------------------------------------------

# Select and rename columns to a clean, consistent schema
scopus_lookup = scopus_long[[
    "issn_norm",
    COL_SOURCE_TITLE,
    COL_TYPE,
    COL_PUBLISHER,
    "active",
    COL_ASJC,
]].drop_duplicates()

# Rename to simpler names for later merging with OpenAlex
scopus_lookup = scopus_lookup.rename(columns={
    COL_SOURCE_TITLE: "source_title",
    COL_TYPE:         "source_type",
    COL_PUBLISHER:    "publisher",
    COL_ASJC:         "asjc_raw",
})

# Add a flag indicating this ISSN is present in the Scopus source list
# (We will later use this to define in_scopus_flag in the works table)
scopus_lookup["scopus_flag"] = True


# ---------------------------------------------------------
# 10. SAVE CLEANED SCOPUS LOOKUP TABLE
# ---------------------------------------------------------

# Save as CSV so we can merge it with OpenAlex works later
scopus_lookup.to_csv("scopus_clean_lookup.csv", index=False)

print("Done! Saved cleaned Scopus lookup as 'scopus_clean_lookup.csv'.")
print("Rows in lookup:", len(scopus_lookup))
print(scopus_lookup.head())


Columns in raw Scopus file:
['Sourcerecord ID', 'Source Title', 'ISSN', 'EISSN', 'Active or Inactive', 'Coverage', 'Titles Discontinued by Scopus Due to Quality Issues', 'Article Language in Source (Three-Letter ISO Language Codes)', 'Medline-sourced Title? (See additional details under separate tab.)', 'Open Access Status', 'Articles in Press Included?', 'Added to List Oct. 2025', 'Source Type', 'Title History Indication', 'Related Title 1', 'Other Related Title 2', 'Other Related Title 3', 'Other Related Title 4', 'Publisher', 'Publisher Imprints Grouped to Main Publisher', 'All Science Journal Classification Codes (ASJC)', 'Top level:\n\nLife Sciences', 'Top level:\n\nSocial Sciences', 'Top level:\n\nPhysical Sciences', 'Top level:\n\nHealth Sciences', '1000 \nGeneral', '1100\nAgricultural and Biological Sciences', '1200\nArts and Humanities', '1300\nBiochemistry, Genetics and Molecular Biology', '1400\nBusiness, Management and Accounting', '1500\nChemical Engineering', '1600\nChemi

In [5]:
with open("scimagojr 2024.csv", "r") as f:
    print(f.readline())
    print(f.readline())
    print(f.readline())


Rank;Sourceid;Title;Type;Issn;Publisher;Open Access;Open Access Diamond;SJR;SJR Best Quartile;H index;Total Docs. (2024);Total Docs. (3years);Total Refs.;Total Citations (3years);Citable Docs. (3years);Citations / Doc. (2years);Ref. / Doc.;%Female;Overton;SDG;Country;Region;Publisher;Coverage;Categories;Areas

1;28773;"Ca-A Cancer Journal for Clinicians";journal;"15424863, 00079235";"John Wiley and Sons Inc";No;No;145,004;Q1;223;43;122;2704;40834;81;168,71;62,88;48,21;4;37;United States;Northern America;"John Wiley and Sons Inc";"1950-2025";"Hematology (Q1); Oncology (Q1)";"Medicine"

2;19434;"MMWR Recommendations and Reports";journal;"10575987, 15458601";"Centers for Disease Control and Prevention (CDC)";Yes;No;41,754;Q1;155;6;15;1652;1308;15;75,11;275,33;75,93;1;5;United States;Northern America;"Centers for Disease Control and Prevention (CDC)";"1990-2024";"Epidemiology (Q1); Health Information Management (Q1); Health (social science) (Q1); Health, Toxicology and Mutagenesis (Q1); Me

In [6]:

# ---------------------------------------------------------
# 1. LOAD RAW SCIMAGO SJR DATA
# ---------------------------------------------------------

# Read the SCImago file (CSV or Excel).
# If your file is Excel, use read_excel instead.

sjr_raw = pd.read_csv( "scimagojr 2024.csv",
    sep=";"
)


In [7]:
# Print columns to verify names (just for checking)
print("Columns in raw SJR file:")
print(sjr_raw.columns.tolist())



Columns in raw SJR file:
['Rank', 'Sourceid', 'Title', 'Type', 'Issn', 'Publisher', 'Open Access', 'Open Access Diamond', 'SJR', 'SJR Best Quartile', 'H index', 'Total Docs. (2024)', 'Total Docs. (3years)', 'Total Refs.', 'Total Citations (3years)', 'Citable Docs. (3years)', 'Citations / Doc. (2years)', 'Ref. / Doc.', '%Female', 'Overton', 'SDG', 'Country', 'Region', 'Publisher.1', 'Coverage', 'Categories', 'Areas']


In [8]:

# ---------------------------------------------------------
# 2. DEFINE COLUMN NAMES WE WILL USE
#    These must match what you see in the print above.
# ---------------------------------------------------------

COL_TITLE        = "Title"              # journal name
COL_TYPE         = "Type"               # journal / book series / etc.
COL_ISSN         = "Issn"               # comma-separated ISSNs
COL_SJR          = "SJR"                # SJR indicator (e.g. "145,004")
COL_SJR_QUARTILE = "SJR Best Quartile"  # e.g. "Q1", "Q2"
COL_HINDEX       = "H index"            # integer


# ---------------------------------------------------------
# 3. FUNCTION TO NORMALIZE ONE ISSN STRING
# ---------------------------------------------------------

def norm_issn(x):
    """
    Normalize a single ISSN:
    - if NaN → return NaN
    - convert to string
    - remove hyphens
    - strip spaces
    - lowercase (so X becomes x)
    """
    if pd.isna(x):
        return np.nan
    return str(x).replace("-", "").strip().lower()


# ---------------------------------------------------------
# 4. SPLIT THE 'Issn' FIELD INTO A LIST OF INDIVIDUAL ISSNs
#    Example: "15424863, 00079235" -> ["15424863", "00079235"]
# ---------------------------------------------------------

def split_issn_list(x):
    """
    Take the raw 'Issn' cell (possibly 'A, B') and return a Python list:
    - if NaN → empty list
    - split by comma
    - strip spaces around each piece
    - drop empty strings
    """
    if pd.isna(x):
        return []
    parts = str(x).split(",")
    clean_parts = [p.strip() for p in parts if p.strip() != ""]
    return clean_parts

# Apply to create a list column
sjr_raw["issn_list_raw"] = sjr_raw[COL_ISSN].apply(split_issn_list)


# ---------------------------------------------------------
# 5. NORMALIZE ALL ISSNs IN THE LIST
#    (Apply norm_issn to each element)
# ---------------------------------------------------------

def normalize_issn_list(lst):
    """
    Take a list of ISSN strings and normalize each:
    - apply norm_issn
    - drop NaNs
    - return unique values (set) as a list
    """
    normed = []
    for x in lst:
        n = norm_issn(x)
        if pd.notna(n):
            normed.append(n)
    # use set() to avoid duplicates, then back to list
    return list(set(normed))

sjr_raw["issn_list_norm"] = sjr_raw["issn_list_raw"].apply(normalize_issn_list)


# ---------------------------------------------------------
# 6. EXPLODE: ONE ROW PER (SOURCE, ISSN)
#    This gives us: issn_norm + SJR for each ISSN.
# ---------------------------------------------------------

sjr_exploded = sjr_raw.explode("issn_list_norm")

# Rename the exploded list element to a simple 'issn_norm' column
sjr_exploded = sjr_exploded.rename(columns={"issn_list_norm": "issn_norm"})

# Drop rows where 'issn_norm' is still missing (no usable ISSN)
sjr_exploded = sjr_exploded.dropna(subset=["issn_norm"])


# ---------------------------------------------------------
# 7. CLEAN / CONVERT SJR VALUE TO FLOAT
# ---------------------------------------------------------

def parse_sjr_value(x):
    """
    Convert the SJR string to float.
    SCImago often uses comma as decimal separator (e.g., "145,004").
    Steps:
    - if NaN or "-" → return NaN
    - replace comma with dot (e.g. '3,456' -> '3.456')
    - convert to float
    """
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "-" or s == "":
        return np.nan
    # replace comma with dot to interpret as decimal
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan

# Create numeric SJR value column
sjr_exploded["sjr_value"] = sjr_exploded[COL_SJR].apply(parse_sjr_value)


# ---------------------------------------------------------
# 8. CLEAN QUARTILE AND H-INDEX
# ---------------------------------------------------------

# Quartile: keep as string; if missing, set to NaN
sjr_exploded["sjr_quartile"] = sjr_exploded[COL_SJR_QUARTILE].replace("-", np.nan)

# H-index: convert to numeric (float or int), set errors to NaN
sjr_exploded["h_index_scimago"] = pd.to_numeric(
    sjr_exploded[COL_HINDEX], errors="coerce"
)


# ---------------------------------------------------------
# 9. BUILD FINAL SJR LOOKUP TABLE (ONE ROW PER ISSN)
#    We keep only the columns we need for merging:
#    issn_norm, sjr_value, sjr_quartile, h_index_scimago
# ---------------------------------------------------------

sjr_lookup = sjr_exploded[[
    "issn_norm",
    "sjr_value",
    "sjr_quartile",
    "h_index_scimago",
]].drop_duplicates()

# Optional: if some ISSN appears multiple times with different SJR values,
# you could aggregate (e.g., take max SJR, best quartile).
# For now, we assume one row per ISSN.


# ---------------------------------------------------------
# 10. OPTIONAL FILTER: KEEP ONLY Q1–Q3 JOURNALS
#     (THIS IS A THRESHOLD YOU CAN APPLY LATER IN STRICT)
# ---------------------------------------------------------

# Example: add a boolean flag "keep_q1_q3" if quartile is Q1, Q2, or Q3
sjr_lookup["keep_q1_q3"] = sjr_lookup["sjr_quartile"].isin(["Q1", "Q2", "Q3"])


# ---------------------------------------------------------
# 11. SAVE CLEANED SJR LOOKUP
# ---------------------------------------------------------

sjr_lookup.to_csv("sjr_clean_lookup.csv", index=False)

print("Done! Saved cleaned SJR lookup as 'sjr_clean_lookup.csv'.")
print("Rows in SJR lookup:", len(sjr_lookup))
print(sjr_lookup.head())

Done! Saved cleaned SJR lookup as 'sjr_clean_lookup.csv'.
Rows in SJR lookup: 50280
  issn_norm  sjr_value sjr_quartile  h_index_scimago  keep_q1_q3
0  00079235    145.004           Q1              223        True
0  15424863    145.004           Q1              223        True
1  15458601     41.754           Q1              155        True
1  10575987     41.754           Q1              155        True
2  14710080     37.353           Q1              531        True


# OpenAlex

In [9]:
import ast
import pandas as pd

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

works = pd.read_csv("open_alex_raw_flagged.csv")

print("Loaded works dataframe:", works.shape)

# ---------------------------------------------------------
# 1. SAFELY PARSE JSON-LIKE COLUMNS
# ---------------------------------------------------------

json_cols = ["all_issns", "topics", "concepts", "locations", "authorships", "primary_topic"]

def safe_parse(x):
    """
    Safely convert list/dict‐like strings into Python objects.
    Prevents NaN errors.
    """
    if not isinstance(x, str):
        return x

    x = x.strip()
    if not x:
        return x

    if x.startswith("[") or x.startswith("{"):
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return x

    return x

for col in json_cols:
    if col in works.columns:
        works[col] = works[col].apply(safe_parse)

print("✓ JSON-like fields parsed successfully")

# ---------------------------------------------------------
# 2. PARSE ANY 'all_issns' STRINGS INTO PYTHON LIST
# ---------------------------------------------------------

def parse_all_issns(cell):
    if isinstance(cell, list):
        return cell
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    if s == "" or s == "[]":
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
        return []
    except (SyntaxError, ValueError):
        return []

works["issn_list"] = works["all_issns"].apply(parse_all_issns)

# ---------------------------------------------------------
# 3. NORMALIZE SINGLE ISSN
# ---------------------------------------------------------

def norm_issn(x):
    if pd.isna(x):
        return None
    s = str(x)
    s = s.replace("-", "")
    s = s.strip()
    s = s.lower()
    if s == "":
        return None
    return s

# ---------------------------------------------------------
# 4. NORMALIZE THE ISSN LIST PER WORK
# ---------------------------------------------------------

def normalize_issn_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for x in lst:
        n = norm_issn(x)
        if n is not None:
            cleaned.append(n)
    seen = set()
    unique = []
    for n in cleaned:
        if n not in seen:
            seen.add(n)
            unique.append(n)
    return unique

works["issn_norm_list"] = works["issn_list"].apply(normalize_issn_list)

works["has_any_issn"] = works["issn_norm_list"].apply(lambda lst: len(lst) > 0)
works["num_issn"] = works["issn_norm_list"].apply(len)

print("✓ ISSN parsing & normalization done")

# ---------------------------------------------------------
# 5. SAVE CLEAN VERSION
# ---------------------------------------------------------

output_path = "openalex_update_cleaned.csv"
works.to_csv(output_path, index=False)

print(f"✓ Saved cleaned & structured OpenAlex dataset:\n{output_path}")


Loaded works dataframe: (9689, 29)
✓ JSON-like fields parsed successfully
✓ ISSN parsing & normalization done
✓ Saved cleaned & structured OpenAlex dataset:
openalex_update_cleaned.csv


In [10]:
# 1) Check how many rows we actually have
print(works.shape)

# 2) Look at the exact row that contains Sheffield text
mask = works.apply(
    lambda col: col.astype(str).str.contains("Sheffield University Management School", na=False)
)
rows_with_sheffield = works[mask.any(axis=1)]

rows_with_sheffield[
    ["all_issns", "issn_list", "issn_norm_list", "has_any_issn", "num_issn"]
]


(9689, 33)


,all_issns,issn_list,issn_norm_list,has_any_issn,num_issn
911,"[02684012, 18734707]","[02684012, 18734707]","[02684012, 18734707]",True,2


# Merging

In [11]:
# ---------------------------------------------
# 1. PARSE issn_norm_list BACK INTO PYTHON LISTS
# ---------------------------------------------

def parse_issn_norm_list(cell):
    """
    Ensure issn_norm_list is a real Python list.

    Cases:
    - If it's already a list: return as is.
    - If NaN / empty / "[]": return [].
    - If it's a string like "['10416080', '18733425']": parse with ast.literal_eval.
    """
    if isinstance(cell, list):
        return cell

    if pd.isna(cell):
        return []

    s = str(cell).strip()
    if s == "" or s == "[]":
        return []

    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
        return []
    except (SyntaxError, ValueError):
        return []

works["issn_norm_list"] = works["issn_norm_list"].apply(parse_issn_norm_list)

# Quick check
print(works[["all_issns", "issn_norm_list"]].head(5))


              all_issns        issn_norm_list
0  [10416080, 18733425]  [10416080, 18733425]
1  [19424795, 19424787]  [19424795, 19424787]
2  [13602357, 15737608]  [13602357, 15737608]
3            [16641078]            [16641078]
4  [15604292, 15604306]  [15604292, 15604306]


In [12]:
# ---------------------------------------------
# 2. EXPLODE issn_norm_list → ONE ISSN PER ROW
# ---------------------------------------------

# Each row becomes multiple rows if it has multiple ISSNs
works_issn = works.explode("issn_norm_list")

# Rename for clarity
works_issn = works_issn.rename(columns={"issn_norm_list": "issn_norm"})

# Drop rows where we have no ISSN at all
works_issn = works_issn.dropna(subset=["issn_norm"])

print("Rows in works_issn (exploded):", len(works_issn))
print(works_issn[["id", "all_issns", "issn_norm"]].head(10))


Rows in works_issn (exploded): 11516
                                 id             all_issns issn_norm
0  https://openalex.org/W4323655724  [10416080, 18733425]  10416080
0  https://openalex.org/W4323655724  [10416080, 18733425]  18733425
1  https://openalex.org/W3000065748  [19424795, 19424787]  19424795
1  https://openalex.org/W3000065748  [19424795, 19424787]  19424787
2  https://openalex.org/W4304943299  [13602357, 15737608]  13602357
2  https://openalex.org/W4304943299  [13602357, 15737608]  15737608
3  https://openalex.org/W4282940252            [16641078]  16641078
4  https://openalex.org/W3155263273  [15604292, 15604306]  15604292
4  https://openalex.org/W3155263273  [15604292, 15604306]  15604306
5  https://openalex.org/W3011958495            [21693536]  21693536


In [13]:
# ---------------------------------------------
# 3. MERGE WITH SCOPUS LOOKUP
# ---------------------------------------------

scopus_cols = ["issn_norm", "source_title", "source_type", "publisher",
               "active", "asjc_raw", "scopus_flag"]

works_issn = works_issn.merge(
    scopus_lookup[scopus_cols],
    on="issn_norm",
    how="left"
)

# ---------------------------------------------
# 3b. MERGE WITH SJR LOOKUP
# ---------------------------------------------

sjr_cols = ["issn_norm", "sjr_value", "sjr_quartile", "h_index_scimago", "keep_q1_q3"]

works_issn = works_issn.merge(
    sjr_lookup[sjr_cols],
    on="issn_norm",
    how="left"
)

print(works_issn[["id", "issn_norm", "source_title", "sjr_quartile", "sjr_value"]].head(10))


                                 id issn_norm  \
0  https://openalex.org/W4323655724  10416080   
1  https://openalex.org/W4323655724  18733425   
2  https://openalex.org/W3000065748  19424795   
3  https://openalex.org/W3000065748  19424787   
4  https://openalex.org/W4304943299  13602357   
5  https://openalex.org/W4304943299  15737608   
6  https://openalex.org/W4282940252  16641078   
7  https://openalex.org/W3155263273  15604292   
8  https://openalex.org/W3155263273  15604306   
9  https://openalex.org/W3011958495  21693536   

                                        source_title sjr_quartile  sjr_value  
0                Learning and Individual Differences           Q1      1.591  
1                Learning and Individual Differences           Q1      1.591  
2  Wiley Interdisciplinary Reviews: Data Mining a...           Q1      2.202  
3  Wiley Interdisciplinary Reviews: Data Mining a...           Q1      2.202  
4             Education and Information Technologies           Q1

In [14]:
# ---------------------------------------------
# 4. AGGREGATE BACK TO WORK LEVEL
# ---------------------------------------------

def any_true(series):
    """Return True if any value in the series is True."""
    return bool(series.fillna(False).any())

def first_non_null(series):
    """Return the first non-null value in a series, or NaN if all null."""
    nonnull = series.dropna()
    return nonnull.iloc[0] if len(nonnull) > 0 else np.nan

def best_quartile(series):
    """
    Choose the best quartile (Q1 is best, then Q2, Q3, Q4).
    If no valid quartile, return NaN.
    """
    order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
    vals = [q for q in series.dropna() if q in order]
    if not vals:
        return np.nan
    return sorted(vals, key=lambda q: order[q])[0]


agg = (
    works_issn
    .groupby("id", as_index=False)
    .agg(
        in_scopus_flag=("scopus_flag", any_true),
        scopus_type=("source_type", first_non_null),
        sjr_quartile=("sjr_quartile", best_quartile),
        sjr_value=("sjr_value", "max"),
        h_index_scimago=("h_index_scimago", "max"),
        keep_q1_q3_any=("keep_q1_q3", any_true),
    )
)

print("Rows in aggregated table (one per work):", len(agg))
print(agg.head(10))


C:\Users\hamza\AppData\Local\Temp\ipykernel_7068\2245579418.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return bool(series.fillna(False).any())


Rows in aggregated table (one per work): 7418
                                 id  in_scopus_flag  scopus_type sjr_quartile  \
0  https://openalex.org/W2182266416           False          NaN          NaN   
1  https://openalex.org/W2182677989           False          NaN          NaN   
2  https://openalex.org/W2185004456           False          NaN          NaN   
3  https://openalex.org/W2600942494            True  Book Series           Q2   
4  https://openalex.org/W2755051337            True      Journal           Q2   
5  https://openalex.org/W2760664994           False          NaN          NaN   
6  https://openalex.org/W2766436289            True      Journal           Q1   
7  https://openalex.org/W2797849812            True      Journal           Q1   
8  https://openalex.org/W2798671091            True  Book Series           Q2   
9  https://openalex.org/W2904980464            True      Journal           Q1   

   sjr_value  h_index_scimago  keep_q1_q3_any  
0        NaN  

In [15]:
# ---------------------------------------------
# 5. MERGE ENRICHMENT BACK TO OPENALEX WORKS
# ---------------------------------------------

works_enriched = works.merge(agg, on="id", how="left")

print(works_enriched[[
    "id", "in_scopus_flag", "scopus_type", "sjr_quartile",
    "sjr_value", "h_index_scimago", "keep_q1_q3_any"
]].head(10))


                                 id in_scopus_flag  scopus_type sjr_quartile  \
0  https://openalex.org/W4323655724           True      Journal           Q1   
1  https://openalex.org/W3000065748           True      Journal           Q1   
2  https://openalex.org/W4304943299           True      Journal           Q1   
3  https://openalex.org/W4282940252           True      Journal           Q2   
4  https://openalex.org/W3155263273           True      Journal           Q1   
5  https://openalex.org/W3011958495           True      Journal           Q1   
6  https://openalex.org/W3156614709           True      Journal           Q1   
7  https://openalex.org/W3087232239           True      Journal           Q1   
8  https://openalex.org/W4367459575           True  Book Series          NaN   
9  https://openalex.org/W3127128771          False          NaN          NaN   

   sjr_value  h_index_scimago keep_q1_q3_any  
0      1.591            113.0           True  
1      2.202             

In [16]:
# ---------------------------------------------
# 6. STRICT FLAGS
# ---------------------------------------------

# Simple STRICT: anything in Scopus
works_enriched["is_strict_in_scopus"] = works_enriched["in_scopus_flag"].fillna(False)

# Stricter: in Scopus AND at least one Q1–Q3 venue
works_enriched["is_strict_q1_q3"] = (
    works_enriched["in_scopus_flag"].fillna(False)
    & works_enriched["keep_q1_q3_any"].fillna(False)
)

# Two strict subsets
strict_any_scopus = works_enriched[works_enriched["is_strict_in_scopus"]].copy()
strict_q1_q3 = works_enriched[works_enriched["is_strict_q1_q3"]].copy()

print("STRICT (any Scopus) rows:", len(strict_any_scopus))
print("STRICT (Scopus & Q1–Q3) rows:", len(strict_q1_q3))


STRICT (any Scopus) rows: 5622
STRICT (Scopus & Q1–Q3) rows: 5064


C:\Users\hamza\AppData\Local\Temp\ipykernel_7068\1832781872.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  works_enriched["is_strict_in_scopus"] = works_enriched["in_scopus_flag"].fillna(False)
C:\Users\hamza\AppData\Local\Temp\ipykernel_7068\1832781872.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  works_enriched["in_scopus_flag"].fillna(False)
C:\Users\hamza\AppData\Local\Temp\ipykernel_7068\1832781872.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call resu

In [17]:
# ---------------------------------------------
# 7. SAVE INTERMEDIATE + FINAL FILES
# ---------------------------------------------

# 1) Exploded view (useful for debugging)
works_issn.to_csv(
    "openalex_works_issn_exploded.csv",
    index=False
)

# 2) Main enriched works table (ALL layer with enrichment)
works_enriched.to_csv(
    "openalex_works_enriched.csv",
    index=False
)

# 3) STRICT subsets (optional)
strict_any_scopus.to_csv(
    "openalex_works_strict_any_scopus.csv",
    index=False
)

strict_q1_q3.to_csv(
    "openalex_works_strict_q1_q3.csv",
    index=False
)

print("Saved: exploded, enriched, and STRICT tables.")


Saved: exploded, enriched, and STRICT tables.


In [18]:
print("====================================")
print("🔍 DATA SIZE QAQC CHECK")
print("====================================")

# 1) Original OpenAlex (before cleaning)
try:
    original = pd.read_csv("openalex_update_1.csv")
    print("Original OpenAlex works:", original.shape)
except:
    print("Original OpenAlex not found / skipped.")

# 2) Cleaned initial version (after ISSN normalization)
try:
    cleaned = pd.read_csv("openalex_update_cleaned.csv")
    print("Cleaned OpenAlex:", cleaned.shape)
except:
    print("Cleaned OpenAlex not found / skipped.")

# 3) Exploded ISSN table
try:
    works_issn = pd.read_csv("openalex_works_issn_exploded.csv")
    print("Exploded ISSN rows:", works_issn.shape)
except:
    print("Exploded ISSN file not found ❗")

# 4) Enriched dataset (ALL works)
try:
    works_enriched = pd.read_csv("openalex_works_enriched.csv")
    print("ALL works enriched:", works_enriched.shape)
except:
    print("Enriched file not found ❗")

# 5) Strict Scopus subset
try:
    strict_any = pd.read_csv("openalex_works_strict_any_scopus.csv")
    print("STRICT: any Scopus:", strict_any.shape)
except:
    print("strict_any_scopus not found ❗")

# 6) Strict Q1–Q3 subset
try:
    strict_q1q3 = pd.read_csv("openalex_works_strict_q1_q3.csv")
    print("STRICT: Q1–Q3:", strict_q1q3.shape)
except:
    print("strict_q1_q3 not found ❗")


🔍 DATA SIZE QAQC CHECK
Original OpenAlex not found / skipped.
Cleaned OpenAlex: (9689, 33)
Exploded ISSN rows: (11516, 43)
ALL works enriched: (9689, 41)
STRICT: any Scopus: (5622, 41)
STRICT: Q1–Q3: (5064, 41)


In [19]:
#print the columns names amd the datatype in the columns of openalex_works_strict_q1_q3
openalex_works_strict_q1_q3 = pd.read_csv("openalex_works_strict_q1_q3.csv")
print("Columns in openalex_works_strict_q1_q3:")
print(openalex_works_strict_q1_q3.dtypes)

Columns in openalex_works_strict_q1_q3:
id                            object
doi                           object
title                         object
abstract_inverted_index       object
publication_year               int64
publication_date              object
open_access                   object
type                          object
language                      object
cited_by_count                 int64
primary_location              object
best_oa_location              object
primary_topic                 object
topics                        object
locations                     object
concepts                      object
authorships                   object
referenced_works              object
countries_distinct_count       int64
keywords                      object
counts_by_year                object
has_eu_affiliation              bool
has_multiple_institutions       bool
distinct_institutions         object
has_multiple_countries          bool
distinct_countries            objec

In [20]:
#read openalex_works_strict_q1_q3 and print the first 5 rows
openalex_works_strict_q1_q3 = pd.read_csv("openalex_works_strict_q1_q3.csv")
openalex_works_strict_q1_q3.head()

,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,has_any_issn,num_issn,in_scopus_flag,scopus_type,sjr_quartile,sjr_value,h_index_scimago,keep_q1_q3_any,is_strict_in_scopus,is_strict_q1_q3
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,True,2,True,Journal,Q1,1.591,113.0,True,True,True
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,True,2,True,Journal,Q1,2.202,79.0,True,True,True
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,True,2,True,Journal,Q1,1.654,97.0,True,True,True
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,True,1,True,Journal,Q2,0.872,212.0,True,True,True
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,True,2,True,Journal,Q1,1.960,68.0,True,True,True
